In [1]:
import drjit as dr
from enum import Enum
import matplotlib.pyplot as plt
import mitsuba as mi
from pathlib import Path
from tqdm import tqdm
from typing import List, Optional, Tuple

mi.set_variant('cuda_ad_rgb')

from mimt import *

class Setting(Enum):
    Sphere       = 1
    Teapot       = 2
    TeapotTwist  = 3
    TeapotMirror = 4
    Spot         = 5
    Bunny        = 6
    Disco        = 7

setting_labels = {
    Setting.Sphere:       "Sphere",
    Setting.Teapot:       "Teapot",
    Setting.TeapotTwist:  "Twist",
    Setting.TeapotMirror: "Mirror",
    Setting.Spot:         "Spot",
    Setting.Bunny:        "Bunny",
    Setting.Disco:        "Disco",
}

In [2]:
def create_scene(setting: Setting, integrator: str, resolution=(256, 256), max_depth: int=5, integrator_args={}):
    scene_dict = mi.cornell_box()

    scene_dict['sensor']['film']['width']  = resolution[1]
    scene_dict['sensor']['film']['height'] = resolution[0]

    scene_dict['integrator']['type']      = integrator
    scene_dict['integrator']['max_depth'] = max_depth
    for (k, v) in integrator_args.items():
        scene_dict['integrator'][k] = v
    scene_dict['sensor']['film']['sample_border'] = True

    # Remove the boxes and add a custom shape
    del scene_dict['large-box']
    del scene_dict['small-box']
    scene_dict['shape'] = {
        'type': 'sphere',
        'radius': 0.5,
        'bsdf': {
            'type': 'roughplastic',
            'diffuse_reflectance': {
                'type': 'rgb',
                'value': [1, 0.6, 0.1]
            }
        }
    }

    # Implement variants of the scene based on the selected setting
    if setting in [Setting.Teapot, Setting.TeapotMirror, Setting.TeapotTwist]:
        del scene_dict['shape']['radius']
        scene_dict['shape']['type'] = 'ply'
        scene_dict['shape']['filename'] = 'data/teapot.ply'
        xangle = -130 if setting == Setting.TeapotTwist else -110
        scene_dict['shape']['to_world'] = mi.ScalarTransform4f().translate([0,-0.3,0]).rotate([0, 1, 0], -150).rotate([1, 0, 0], xangle).scale(0.25)
    if setting == Setting.TeapotMirror:
        # Looking up to the ceiling
        scene_dict['sensor']['to_world'] = mi.ScalarTransform4f().look_at(
            origin=[-0.5, -1.1, 2.4],
            target=[0.2, 0.6, 0.],
            up=[0, 1, 0]
        )
        # Mirror-like ceiling and teapot
        scene_dict['ceiling']['bsdf'] = {'type': 'roughconductor', 'alpha': 0.02}
        scene_dict['shape']['bsdf']   = {'type': 'roughconductor', 'alpha': 0.02}
        scene_dict['green']['reflectance'] = { 'type': 'checkerboard',
                                               'to_uv': mi.ScalarTransform3f().scale(3),
                                               'color0': {'type':'rgb', 'value':[0.105421, 0.37798, 0.076425]},
                                               'color1': {'type':'rgb', 'value':[0.1, 0.1, 0.1]}}
    if setting == Setting.Bunny:
        del scene_dict['shape']['radius']
        scene_dict['shape']['type'] = 'obj'
        scene_dict['shape']['filename'] = 'data/bunny.obj'
        scene_dict['shape']['to_world'] = mi.ScalarTransform4f().translate([-0.05, -0.45, 0]).scale(0.25) # .rotate([1, 0, 0], -20)
        scene_dict['shape']['bsdf']     = {'type': 'roughconductor', 'alpha': 0.3}
        scene_dict['light']['to_world'] = mi.ScalarTransform4f(scene_dict['light']['to_world']).scale(0.5)
        scene_dict['light']['emitter']['radiance']['value'] = 2*np.array(scene_dict['light']['emitter']['radiance']['value'])
        scene_dict['light2'] = scene_dict['light'].copy()
        scene_dict['light2']['to_world'] = mi.ScalarTransform4f().look_at(origin=[0.6, 0.1, 0.6], target=[2, 0, 0], up=[0, 1, 0]).scale([0.23, 0.19, 0.19]).scale(0.5)
        scene_dict['green-wall']['bsdf'] = {
            'type': 'diffuse',
            'reflectance': { 
                'type': 'checkerboard',
                'to_uv': mi.ScalarTransform3f().scale(3),
                'color0': {'type':'rgb', 'value':[0.105421, 0.37798, 0.076425]},
                'color1': {'type':'rgb', 'value':[0.37798, 0.37798, 0.076425]}
            }
        }
    if setting == Setting.Disco:
        #scene_dict['shape']['to_world'] = mi.ScalarTransform4f().translate([-0.05, 0.75, 0]).scale(0.25)
        bayer = dr.zeros(mi.TensorXf, (16, 16, 3))
        bayer[ ::2,  ::2, 2] = 1.
        bayer[ ::2, 1::2, 1] = 1.
        bayer[1::2, 1::2, 0] = 1.
        scene_dict['shape']['bsdf']['diffuse_reflectance'] = {
            'type': 'bitmap',
            'bitmap': mi.Bitmap(bayer),
            'raw': True,
            'filter_type': 'nearest'
        }
        scene_dict['light']['emitter']['radiance']['value'] = 2*np.array(scene_dict['light']['emitter']['radiance']['value'])
        scene_dict['light']['to_world'] = mi.ScalarTransform4f().translate([0.0, 0.01, -0.99]).scale([0.23, 0.19, 0.19])
    if setting == Setting.Spot:
        del scene_dict['shape']['radius']
        scene_dict['shape']['type'] = 'obj'
        scene_dict['shape']['filename'] = 'data/spot.obj'
        scene_dict['shape']['to_world'] = mi.ScalarTransform4f().translate([-0.2, -0.4, 0.2]).rotate([0, 1, 0], 65).scale(0.8)

        scene_dict['light']['emitter']['radiance']['value'] = 2*np.array(scene_dict['light']['emitter']['radiance']['value'])
        scene_dict['light']['to_world'] = mi.ScalarTransform4f().translate([0.0, 0.01, -0.99]).scale([0.23, 0.19, 0.19])

        scene_dict['mirror1'] = {
            'type': 'rectangle',
            'to_world': mi.ScalarTransform4f().look_at(origin=[0.6, 0.6, -0.6], target=[0, 0, 0.8], up=[0, 1, 0]).scale(0.4),
            'bsdf': {'type': 'roughconductor', 'alpha': 0.025}
        }

    return scene_dict

def translate_to_world(params: mi.SceneParameters, key: str, original: mi.Transform4f, offset: mi.Vector3f):
    params[f'{key}.to_world'] = mi.Transform4f().translate(offset) @ original
    params.update()
    return params

def translate_vertices(params: mi.SceneParameters, key: str, original: Tuple[mi.Float, mi.Float], offset: mi.Vector3f):
    vertex_positions_original = original[0]
    vertex_normals_original   = original[1]
    transform = mi.Transform4f().translate(offset)
    params[f'{key}.vertex_positions'] = dr.ravel(transform.transform_affine(dr.unravel(mi.Point3f,  vertex_positions_original)))
    params[f'{key}.vertex_normals']   = dr.ravel(transform.transform_affine(dr.unravel(mi.Vector3f, vertex_normals_original)))
    params.update()
    return params

In [3]:
def generate_data(settings: List[Setting], integrators: List[str], spp: int, spp_grad: int, spp_fd: int, grad_passes: int = 1, fd_passes: int = 1, fd_shared: bool = True, grad_skip_first: bool = False, integrators_args: Optional[List] = None):
    data = {}

    for setting in settings:
        # Generate the data for the gi
        data[setting_labels[setting]] = []
        grad_fd_shared = None
        for integrator_idx, integrator in enumerate(pbar:= tqdm(integrators, leave=False, desc="integrator")):
            tqdm.write("    "+integrator)
            pbar.set_postfix_str(integrator)
            intergrator_args = {} if integrators_args is None else integrators_args[integrator_idx]
            scene_dict = create_scene(setting=setting, integrator=integrator.lower(), integrator_args=intergrator_args)
            scene = mi.load_dict(scene_dict)

            params   = mi.traverse(scene)
            if setting in [Setting.Sphere, Setting.Disco]:
                original = mi.Transform4f(params['shape.to_world'])
                def offset(x):
                    return [0, 0, -x] if setting == Setting.Sphere else [x, 0, 0]
                apply_transform = lambda x: translate_to_world(params, 'shape', original, offset(x))
            else:
                original = (mi.Float(params['shape.vertex_positions']), mi.Float(params['shape.vertex_normals']))
                apply_transform = lambda x: translate_vertices(params, 'shape', original, [0, 0, -x])

            # Render primal image
            img = mi.render(scene, params=params, seed=10, spp=spp, spp_grad=spp_grad)

            # Compute gradients
            spp_fd = spp_fd if spp_fd > 0 else spp_grad
            if fd_shared and grad_fd_shared is not None:
                grad_fd = grad_fd_shared
            else:
                grad_fd = 0
                with dr.suspend_grad():
                    for pass_idx in range(fd_passes):
                        render_with_offset = lambda x: mi.render(scene, params=apply_transform(x), seed=pass_idx, spp=spp_fd)
                        grad_fd += compute_gradient_finite_differences(render_with_offset, 0., h=0.001)
                grad_fd /= fd_passes
                grad_fd_shared = grad_fd

            if grad_skip_first and integrator_idx == 0:
                grad_fw = dr.detach(grad_fd * 0) # Insert dummy gradient, not used
            else:
                grad_fw = 0
                seed_offset = 0
                for pass_idx in range(grad_passes):
                    render_with_offset = lambda x: mi.render(scene, params=apply_transform(x), seed=pass_idx+seed_offset, spp=spp, spp_grad=spp_grad)

                    # In extremely rare cases, the implementation produces NaNs.
                    # Guard against them by re-running with different seed until the pass succeeds
                    success = False
                    while not success:
                        grad_fw_current = compute_gradient_forward(render_with_offset, 0.)
                        if dr.all(dr.isfinite(grad_fw_current)):
                            success = True
                        else:
                            print(f"NaN {integrator}, pass={pass_idx}")
                            seed_offset += 1

                    grad_fw += grad_fw_current

            
            dr.eval(img, grad_fd, grad_fw)
            mi.Bitmap(img).write('primal.exr')

            data[setting_labels[setting]].append([integrator, dr.detach(img), grad_fd, grad_fw])

            del params, scene

    return data

# Figure 4: Biased interior derivatives in RB-style path space differentiable rendering

In [4]:
spp_factor  = 8192
integrators = ['prb', 'prb_2', ]
data_4 = generate_data([Setting.Sphere, Setting.Teapot, Setting.Bunny], integrators, spp=4*spp_factor, spp_grad=spp_factor, spp_fd=1,
                       fd_passes=1, grad_passes=1, fd_shared=True, grad_skip_first=False)

integrator:   0%|         | 0/2 [00:00<?, ?it/s, prb]

    prb


integrator:  50%|▌| 1/2 [00:11<00:11, 11.02s/it, prb_

    prb_2


    prb


RuntimeError: ​[PLYMesh] Error while loading PLY file "teapot.ply": file not found!

In [ ]:
generate_figure(integrators, data_4, Path("output/case_2_forward.pdf"), grad_projection='mean', square_r_setting_3=False, quantile=0.99,
                exclude_first_gradient=False)

# Figure 5: Pose Estimation

In [ ]:
def generate_data_pose_estimation(integrators: List[str], num_iterations: int, spp_ref: int, spp_opt: int, spp_opt_grad: Optional[int] = None):
    num_runs = 6
    data = {}
    for run_index in range(num_runs):
        run_key = f'pose_{run_index}'
        data[run_key] = [] # .append((integrator, img, grad_fd, grad_fw))
        for integrator in integrators:
            scene_dict = create_scene(Setting.Teapot, integrator=integrator, resolution=(128, 128))
            scene  = mi.load_dict(scene_dict)
            params = mi.traverse(scene)

            vertex_positions_original = mi.Float(params['shape.vertex_positions'])
            vertex_normals_original   = mi.Float(params['shape.vertex_normals'])

            img_ref = mi.render(scene, spp=spp_ref)
            # mi.Bitmap(img_ref)

            rng = np.random.default_rng(run_index)

            opt = mi.ad.Adam(lr=1e-2)
            t = 0.25 # 0.2
            opt['offset'] = t*mi.Point3f(0.5*(2*rng.random(3)-1))
            opt['quat']   = (1-t)*mi.Quaternion4f(1) + t*mi.Quaternion4f(2*rng.random(4)-1)

            losses = []
            for it in range(num_iterations):
                q_opt = dr.normalize(opt['quat'])
                R_opt = dr.quat_to_matrix(q_opt)
                transform_opt = mi.Transform4f().translate(opt['offset']) @ mi.Transform4f(R_opt)

                params['shape.vertex_positions'] = dr.ravel(transform_opt.transform_affine(dr.unravel(mi.Point3f, vertex_positions_original)))
                params['shape.vertex_normals']   = dr.ravel(transform_opt.transform_affine(dr.unravel(mi.Vector3f, vertex_normals_original)))
                params.update()

                img_opt = mi.render(scene, params=params, spp=2*spp_opt, spp_grad=spp_opt, seed=it)

                loss = dr.mean(dr.abs(img_opt - img_ref))
                dr.backward(loss)
                opt.step()

                if it == 0:
                    with dr.suspend_grad():
                        img_init = mi.render(scene, spp=spp_ref, seed=it)

                dr.print(f"[{run_index+1}/{num_runs}, {integrator}] Iteration {it}, Loss {loss}", end='\r')
                losses.append(dr.detach(loss))

            # Re-render optimized image with reference spp
            with dr.suspend_grad():
                img_opt = mi.render(scene, spp=spp_ref, seed=it)

            dr.eval(img_ref, img_init, img_opt, *losses)
            del params, scene

            data[run_key].append((integrator, img_ref, img_init, img_opt, losses))
    return data

data_pose = generate_data_pose_estimation(integrators=['prb_threepoint', 'prb_projective'], num_iterations=200, spp_ref=128, spp_opt=16)

In [ ]:
import matplotlib.patheffects as pe

plt.rc('axes', prop_cycle=plt.style.library['seaborn-v0_8']['axes.prop_cycle'])

num_integrators = len(list(data_pose.values())[0])

n_rows = len(list(data_pose.keys()))
n_cols = 1+num_integrators+1
aspect = n_rows/n_cols
fig = plt.figure(figsize=(FIGURE_WIDTH_ONE_COLUMN, aspect*FIGURE_WIDTH_ONE_COLUMN))

gs = fig.add_gridspec(n_rows, n_cols, wspace=0.015, hspace=0.02)

num_runs = n_rows
for run_idx, (_, run_data) in enumerate(data_pose.items()):
    img_ref = run_data[0][2]
    ax = disable_ticks(fig.add_subplot(gs[run_idx, 0]))
    ax.imshow(mi.Bitmap(img_ref).convert(srgb_gamma=True), extent=(0, 1, 0, 1))
    if run_idx == num_runs-1:
        ax.set_xlabel("Initial", fontsize=12)

    ax_loss = fig.add_subplot(gs[run_idx, -1])
    ax_loss.set_ylim(ymin=0.008, ymax=0.025)
    ax_loss.set_yticks([])
    ax_loss.yaxis.tick_right()
    ax_loss.yaxis.set_label_position("right")
    ax_loss.set_facecolor('#EAEAF2')
    if run_idx == 0:
        ax_loss.set_ylabel("Loss", fontsize=12)
    if run_idx != num_runs - 1:
        ax_loss.set_xticks([])
    else:
        ax_loss.set_xlabel("Iteration", fontsize=10)
    for integrator_idx, integrator_data in enumerate(run_data):
        img_opt = integrator_data[3]
        ax = disable_ticks(fig.add_subplot(gs[run_idx, 1 + integrator_idx]))
        ax.imshow(mi.Bitmap(img_opt).convert(srgb_gamma=True), extent=(0, 1, 0, 1))
        if run_idx == num_runs-1:
            ax.set_xlabel(integrator_data[0], fontsize=12)

        losses = [l.numpy() for l in integrator_data[4]]
        print(np.min(losses), np.max(losses))
        ax_loss.plot(losses, label=integrator_data[0])
    if run_idx == 0:
        ax_loss.legend(fontsize=8)

fig.savefig("output/case_2_pose.pdf", facecolor='white', bbox_inches='tight', dpi=300)

# Figure 6: Projective Sampling "Fix"

### a)

In [ ]:
spp_factor = 2048
integrators      = ['prb', 'prb_projective_fix', 'prb_projective_fix', 'prb_projective_fix']
integrators_args = [{}, {'include_boundary': False}, {'include_interior': False}, {}]
data_6a = generate_data([Setting.TeapotTwist, Setting.Sphere, Setting.TeapotMirror], integrators, spp=4*spp_factor, spp_grad=4*spp_factor, spp_fd=8*spp_factor,
                        fd_passes=16, grad_passes=20, fd_shared=True, grad_skip_first=True,
                        integrators_args=integrators_args)

In [ ]:
generate_figure(integrators, data_6a, Path("output/case_2_projective_fix_a.pdf"), grad_projection='mean', square_r_setting_3=False, quantile=[0.99, 0.99, 0.97], 
                labels=['Interior\n(RB, ours)', 'Boundary\n(Proj. Sampling)', 'Interior +\nBoundary'], exclude_first_gradient=True)

### b)

In [ ]:
spp_factor = 2048
integrators      = ['prb', 'prb_projective_fix', 'prb_projective_fix', 'prb_projective_fix']
integrators_args = [{}, {'include_boundary': False}, {'include_interior': False}, {}]
data = generate_data([Setting.Disco, Setting.Bunny, Setting.Spot], integrators, spp=4*spp_factor, spp_grad=4*spp_factor, spp_fd=8*spp_factor,
                     fd_passes=16, grad_passes=20, fd_shared=True, grad_skip_first=True,
                     integrators_args=integrators_args)

In [ ]:
generate_figure(integrators, data, Path("output/case_2_projective_fix_b.pdf"), grad_projection='mean', square_r_setting_3=False, quantile=0.99, 
                labels=['Interior\n(RB, ours)', 'Boundary\n(Proj. Sampling)', 'Interior +\nBoundary'], exclude_first_gradient=True)